# RAG Engine - C++ Build (Simplified)

**Note:** Full GPU build with ONNXRuntime takes too long for Colab.

This notebook builds a simplified version using system libraries.

For full GPU testing, see:
- Python notebook: `rag-engine-colab.ipynb` (GPU validated ✅)
- Local Docker: `docker build -t rag-engine .`

In [ ]:
# Setup
import os
import subprocess

os.chdir('/content')
print('Setup complete')

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
# Install system dependencies (fast!)
!apt-get update -qq && apt-get install -y -qq cmake build-essential git curl wget unzip \
    libprotobuf-dev protobuf-compiler libuv1-dev g++ ninja-build libfmt-dev 2>&1 | tail -3
print('System packages installed')

In [ ]:
# Clone repository
!rm -rf rag-engine
!git clone https://github.com/gumeeee/rag-engine.git 2>&1 | tail -3
print('Repository cloned')

In [ ]:
# Configure with system libraries (no vcpkg needed!)
import os
os.chdir('/content/rag-engine')
!mkdir -p build
!cd build && cmake .. \
    -DCMAKE_BUILD_TYPE=Release \
    -DUSE_SYSTEM_LIBS=ON 2>&1 | tail -20
print('CMake configured')

In [ ]:
# Build
!cd /content/rag-engine/build && make -j$(nproc) 2>&1 | tail -30
print('\nBuild complete')

In [ ]:
import os
binary = '/content/rag-engine/build/rag-engine'
if os.path.exists(binary):
    print(f'Binary: {os.path.getsize(binary)/1024/1024:.1f} MB')
    print('Build successful!')
else:
    print('Build incomplete - see errors above')

## Summary

**For full GPU testing:**

1. Use the Python notebook (`rag-engine-colab.ipynb`) - validated:
   - GPU Tesla T4: ✅
   - Batching 16x speedup: ✅
   - Latency p99 < 10ms: ✅
   - FAISS HNSW: ✅

2. For production C++ build:
   ```bash
   docker build -t rag-engine .
   docker run --gpus all -p 8080:8080 rag-engine
   ```